# External Auditors Pipeline v3.2 (Complete & Patched)

## Independent Validation of Job Classifications

**All Colleague Feedback Implemented:**
- ✅ Multi-level role mapping (major role + functional subdomain)
- ✅ Integer confidence scores (1-100) for better variance
- ✅ Seniority consistency validation
- ✅ Provenance tracking FIXED (model names + timestamps)

**Version:** 3.2_Complete_Patched
**Date:** January 2026

## 1. Setup & Configuration

In [ ]:
# Install required packages (run once)
!pip install transformers torch anthropic pandas numpy tqdm ipywidgets --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.3 MB/s eta 0:00:00


In [ ]:
# Install required packages (run once)
!pip install openai anthropic pandas numpy tqdm -q

import pandas as pd
import numpy as np
import json
import os
import re
from datetime import datetime
from typing import Dict, List, Optional
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# For Auditor 1: GPT-4o via OpenAI
import openai

# For Auditor 2: Claude via Anthropic
import anthropic

print("✓ Libraries imported successfully")


✓ Libraries imported successfully


### Configuration Settings

In [ ]:
# =============================================================================
# CONFIGURATION - ENHANCED VERSION
# =============================================================================

# Auditor 1 Configuration (GPT-4o via OpenAI API)
AUDITOR1_MODEL = "gpt-4o"
AUDITOR1_TEMPERATURE = 0.1

# Auditor 2 Configuration (Claude via Anthropic API) - ENHANCED
AUDITOR2_MODEL = "claude-sonnet-4-20250514"
AUDITOR2_TEMPERATURE = 0.4  # INCREASED from 0.3 for better confidence variance

# Input/Output Paths
INPUT_JD_FILE = "Sample_JDs.csv"
ROLE_GROUPS_FILE = "MNPS_Role_Groups.json"
OUTPUT_DIR = "outputs"

# Processing Settings
BATCH_SIZE = 10
SAVE_INTERMEDIATE = True

# Notebook version for provenance
NOTEBOOK_VERSION = "3.2_Complete_Patched"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✓ Configuration loaded (Enhanced & Patched Version)")
print(f"  - Auditor 1 Model: {AUDITOR1_MODEL}")
print(f"  - Auditor 2 Model: {AUDITOR2_MODEL}")
print(f"  - Auditor 2 Temperature: {AUDITOR2_TEMPERATURE} (increased for variance)")
print(f"  - Output directory: {OUTPUT_DIR}")
print(f"  - Notebook version: {NOTEBOOK_VERSION}")

✓ Configuration loaded (Enhanced & Patched Version)
  - Auditor 1 Model: gpt-4o
  - Auditor 2 Model: claude-sonnet-4-20250514
  - Auditor 2 Temperature: 0.4 (increased for variance)
  - Output directory: outputs
  - Notebook version: 3.2_Complete_Patched


In [ ]:
# Load API Keys from Google Colab Secrets
try:
    from google.colab import userdata

    # Get both API keys
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

    print("✓ API keys loaded from Colab secrets")
    print(f"  - OPENAI_API_KEY: {OPENAI_API_KEY[:10]}...")
    print(f"  - ANTHROPIC_API_KEY: {ANTHROPIC_API_KEY[:10]}...")

except ImportError:
    print("⚠ Not running in Google Colab")
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
    ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

    if not OPENAI_API_KEY:
        print("  ⚠ OPENAI_API_KEY not found")
    if not ANTHROPIC_API_KEY:
        print("  ⚠ ANTHROPIC_API_KEY not found")

except Exception as e:
    print(f"⚠ Error loading secrets: {e}")
    OPENAI_API_KEY = ''
    ANTHROPIC_API_KEY = ''


✓ API keys loaded from Colab secrets
  - OPENAI_API_KEY: sk-proj-YI...
  - ANTHROPIC_API_KEY: sk-ant-api...


### Load MNPS Role Group Definitions

In [ ]:
# Default MNPS Role Groups (concise version for Auditor 2)
# Modify this dictionary or load from JSON file

DEFAULT_ROLE_GROUPS = {
    "Accountant": "Financial accounting, budget management, and fiscal reporting",
  "Administrative Assistant": "Administrative support, scheduling, office coordination, and clerical duties",
  "Advisor": "Provides guidance and strategic advice to students, staff, or programs",
  "Agent": "Represents the district in specific transactions or relationships",
  "Aide": "Provides support assistance in classrooms or other educational settings",
  "Analyst": "Data analysis, research, and strategic decision support using quantitative and qualitative methods",
  "Architect (Facility-Focused)": "Designs and plans physical facilities, buildings, and infrastructure",
  "Architect (Technology-Focused)": "Designs and plans technology systems, platforms, and technical infrastructure",
  "Assistant": "Provides general support and assistance in administrative or operational tasks",
  "Assistant Principal": "School-level leadership supporting the principal in building operations and student management",
  "Associate": "Professional support roles assisting in program delivery, operations, or specialized functions",
  "Auditor": "Conducts compliance reviews, financial audits, and quality assurance assessments",
  "Budget Partner": "Collaborates on budget planning, allocation, and fiscal oversight",
  "Buyer": "Procurement and purchasing of goods and services for the district",
  "Cashier": "Handles financial transactions, cash management, and payment processing",
  "Chef": "Plans, prepares, and manages food service operations",
  "Chief": "Executive-level leadership with district-wide strategic responsibility and executive team membership",
  "Clerk": "Performs routine clerical and administrative record-keeping tasks",
  "Coach": "Supports teachers or staff in professional development, instructional strategies, and skill improvement",
  "Coordinator": "Coordinates programs, initiatives, or services across schools or departments with operational focus",
  "Counselor": "Provides counseling services to students for academic, social-emotional, or career guidance",
  "Dean": "Mid-level academic or student affairs leadership, often overseeing specific programs or student populations",
  "Deputy Chief": "Senior leadership role supporting a Chief in district-wide strategic functions",
  "Designer": "Creates visual, instructional, or technical designs and materials",
  "Developer": "Develops programs, curriculum, technology solutions, or organizational capabilities",
  "Director": "Leads departments or major functions with strategic planning, policy development, and multi-team leadership",
  "Dispatcher": "Coordinates and dispatches transportation, services, or resources",
  "Driver": "Operates vehicles for student transportation or district logistics",
  "Engineer": "Designs, builds, or maintains technical systems, facilities, or infrastructure",
  "Executive Director": "Senior leadership role directing major organizational functions or divisions",
  "Executive Officer": "High-level administrative role supporting executive leadership and operations",
  "Facilitator": "Guides processes, meetings, or learning experiences to achieve specific outcomes",
  "Foreman": "Supervises skilled labor crews in facilities, maintenance, or construction work",
  "Instructor": "Provides direct instruction or training in specialized areas",
  "Intern": "Temporary position for professional learning and development",
  "Interpreter": "Provides language interpretation services for communication with non-English speakers",
  "Liaison": "Facilitates communication and coordination between departments, schools, or external partners",
  "Librarian": "Manages library resources, promotes literacy, and supports research and information access",
  "Manager": "Manages teams, projects, or programs with budget oversight and implementation responsibilities",
  "Mechanic": "Repairs and maintains vehicles, equipment, or mechanical systems",
  "Monitor": "Oversees student activities, safety, or compliance with procedures",
  "Officer": "Enforcement, safety, or compliance role with designated authority",
  "Operator": "Operates technical equipment, machinery, or systems",
  "Paraprofessional": "Provides classroom or instructional support under teacher supervision",
  "Pathologist (Speech-Language Focused)": "Assesses and treats speech, language, and communication disorders",
  "Principal": "Building-level leader with authority over all school operations, staff, and student outcomes",
  "Psychologist": "Provides psychological assessment, intervention, and consultation services",
  "Receptionist": "Greets visitors, answers phones, and manages front office operations",
  "Registrar": "Manages student records, enrollment, and academic registration processes",
  "Representative": "Represents the district or specific programs to external stakeholders",
  "Secretary": "Provides executive administrative support to leaders or departments",
  "Skilled Laborer": "Performs skilled trades work in facilities, maintenance, or construction",
  "Social Worker": "Provides social-emotional support, intervention, and family/community connections",
  "Specialist": "Specialized support roles requiring specific expertise in technical, educational, or professional domains",
  "Supervisor": "First-line supervision of staff with operational oversight and performance management",
  "Teacher": "Classroom instruction and direct student learning facilitation in K-12 settings",
  "Technician": "Provides technical support, maintenance, or operation of specialized equipment or systems",
  "Therapist": "Provides therapeutic services (occupational, physical, or other clinical interventions)",
  "Trainer": "Designs and delivers professional development and training programs",
  "Translator": "Translates written materials between languages",
  "Tutor": "Provides individualized or small-group academic instruction and support",
  "Worker": "General labor or support work in facilities, operations, or services",
  "Writer": "Creates written content, communications, or instructional materials"
}

# Try to load from file, otherwise use defaults
try:
    with open(ROLE_GROUPS_FILE, 'r') as f:
        ROLE_GROUPS = json.load(f)
    print(f"✓ Loaded {len(ROLE_GROUPS)} role groups from {ROLE_GROUPS_FILE}")
except FileNotFoundError:
    ROLE_GROUPS = DEFAULT_ROLE_GROUPS
    print(f"⚠ {ROLE_GROUPS_FILE} not found, using default role groups")
    print(f"  Creating {ROLE_GROUPS_FILE} with defaults...")
    with open(ROLE_GROUPS_FILE, 'w') as f:
        json.dump(ROLE_GROUPS, f, indent=2)

# Display role groups
print(f"\nMNPS Role Groups ({len(ROLE_GROUPS)} total):")
for role, desc in ROLE_GROUPS.items():
    print(f"  • {role}: {desc[:80]}..." if len(desc) > 80 else f"  • {role}: {desc}")

✓ Loaded 63 role groups from MNPS_Role_Groups.json

MNPS Role Groups (63 total):
  • Accountant: Financial accounting, budget management, and fiscal reporting
  • Administrative Assistant: Administrative support, scheduling, office coordination, and clerical duties
  • Advisor: Provides guidance and strategic advice to students, staff, or programs
  • Agent: Represents the district in specific transactions or relationships
  • Aide: Provides support assistance in classrooms or other educational settings
  • Analyst: Data analysis, research, and strategic decision support using quantitative and q...
  • Architect (Facility-Focused): Designs and plans physical facilities, buildings, and infrastructure
  • Architect (Technology-Focused): Designs and plans technology systems, platforms, and technical infrastructure
  • Assistant: Provides general support and assistance in administrative or operational tasks
  • Assistant Principal: School-level leadership supporting the principal in buildi

## 2. Data Preparation

In [ ]:
def load_and_prepare_jds(filepath: str, encoding: str = 'latin1') -> pd.DataFrame:
    """
    Load job descriptions and prepare for auditing.

    Args:
        filepath: Path to CSV file with job descriptions
        encoding: File encoding (default: latin1 for Sample_JDs.csv)

    Returns:
        DataFrame with prepared job descriptions
    """
    # Load data
    df = pd.read_csv(filepath, encoding=encoding)

    # Create combined job description text (excluding title fields)
    text_columns = ['Position Summary', 'Education', 'Work Experience',
                   'Essential Functions', 'Licenses and Certifications',
                   'Knowledge, Skills and Abilities']

    # Combine available text columns
    available_cols = [col for col in text_columns if col in df.columns]

    df['combined_jd_text'] = df[available_cols].apply(
        lambda row: '\n\n'.join([
            f"{col}:\n{str(val)}"
            for col, val in zip(available_cols, row)
            if pd.notna(val) and str(val).strip() != ''
        ]),
        axis=1
    )

    # Keep original title for reference only (won't be used in auditing)
    if 'Job Description Name' in df.columns:
        df['original_title'] = df['Job Description Name']

    print(f"✓ Loaded {len(df)} job descriptions")
    print(f"  - Columns used: {', '.join(available_cols)}")
    print(f"  - Average text length: {df['combined_jd_text'].str.len().mean():.0f} characters")

    return df

# Load data
jd_df = load_and_prepare_jds(INPUT_JD_FILE)
print(f"\nSample job description preview (first 500 chars):")
print(jd_df['combined_jd_text'].iloc[0][:500] + "...")

✓ Loaded 67 job descriptions
  - Columns used: Position Summary, Education, Work Experience, Essential Functions, Licenses and Certifications, Knowledge, Skills and Abilities
  - Average text length: 2290 characters

Sample job description preview (first 500 chars):
Position Summary:
Coordinates with the daily transactional review of compensation-related inquiries and requests.  Recommends, develops and implements an appropriate employee compensation structure.  Configures job families and job tracks.  Evaluates new job titles and conducts ongoing reclassification.  Responds to salary surveys.  Coordinates special projects such as: upgrades to the HRTMS job description database software, Market Pay salary survey software, and GGS global grading software; en...


## 3. Auditor 1: GPT-4o (API) - Title Extraction

**Why GPT-4o via API:**
- ✅ Excellent at semantic extraction and instruction-following
- ✅ Consistent, high-quality output with JSON mode
- ✅ Very fast (~1-2 seconds per job)
- ✅ Cost-effective (~$0.01 per 100 jobs)
- ✅ No GPU or local setup required
- ✅ Highly reliable with strong uptime


In [ ]:
# =============================================================================
# AUDITOR 1: GPT-4o TITLE EXTRACTOR (API)
# =============================================================================

class GPT4oAuditor:
    """
    Auditor 1: Uses GPT-4o via OpenAI API to extract job titles.

    Optimized for precise semantic extraction with functional qualifiers.
    Fast, reliable, and cost-effective (~$0.01 per 100 jobs).
    """

    def __init__(self, api_key: str, model: str = "gpt-4o", temperature: float = 0.1):
        """
        Initialize GPT-4o auditor.

        Args:
            api_key: OpenAI API key
            model: OpenAI model identifier (gpt-4o, gpt-4o-mini)
            temperature: Sampling temperature (lower = more deterministic)
        """
        print(f"Initializing GPT-4o Title Auditor: {model}...")

        if not api_key:
            raise ValueError("OPENAI_API_KEY is required but not provided")

        self.client = openai.OpenAI(api_key=api_key)
        self.model = model
        self.temperature = temperature

        print(f"✓ GPT-4o auditor initialized")

    def generate_title(self, job_description: str) -> Dict:
        """
        Extract a market-recognizable job title from description.

        Args:
            job_description: Full job description text

        Returns:
            Dict with suggested_title, confidence_score, reasoning, title_quality
        """
        system_prompt = "You are an expert at extracting precise, market-standard job titles from job descriptions. Always respond with valid JSON."

        user_prompt = f"""Task: Extract a precise, market-standard job title from this job description.

CRITICAL RULES (follow exactly):
1. NEVER return single-word generic labels: "Analyst", "Specialist", "Manager", "Coordinator", "Director"
2. ALWAYS include a functional domain qualifier:
   ✓ GOOD: "Compensation Analyst", "IT Specialist", "Finance Manager", "Health Information Specialist"
   ✗ BAD: "Analyst", "Specialist", "Manager"
3. Use ONLY the duties and responsibilities - completely ignore any job title mentioned in the text
4. Choose the most commonly used industry-standard title for these specific duties
5. Be specific about the functional area based on the work performed

EXAMPLES:
- Duties about compensation structures and pay analysis → "Compensation Analyst" NOT "Analyst"
- Duties about processing vendor invoices → "Accounts Payable Specialist" NOT "Specialist"
- Duties about student occupational therapy → "Occupational Therapist" NOT "Therapist"
- Duties about behavior intervention plans → "Behavior Specialist" NOT "Specialist"

JOB DESCRIPTION:
{job_description[:4000]}

OUTPUT FORMAT (valid JSON only, no markdown):
{{
    "suggested_title": "Specific Job Title With Functional Qualifier",
    "confidence_score": 0.85,
    "reasoning": "Brief explanation of why this title fits the duties"
}}"""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=self.temperature,
                response_format={"type": "json_object"}  # Force JSON output
            )

            response_text = response.choices[0].message.content

            # Parse JSON
            result = json.loads(response_text.strip())

            # Validate title quality
            result['title_quality'] = self._validate_title_quality(result.get('suggested_title', ''))
            result['model'] = self.model

            return result

        except json.JSONDecodeError as e:
            print(f"⚠ JSON parsing error: {e}")
            print(f"  Raw response (first 300 chars): {response_text[:300] if 'response_text' in locals() else 'N/A'}")
            return {
                "suggested_title": "Error",
                "confidence_score": 0.0,
                "model": self.model,
                "reasoning": f"JSON parse error: {str(e)}",
                "title_quality": "error"
            }
        except Exception as e:
            error_msg = str(e)
            print(f"⚠ API error: {type(e).__name__}: {error_msg}")

            # Provide helpful diagnostics
            if "401" in error_msg or "unauthorized" in error_msg.lower():
                print("  → Check your OPENAI_API_KEY in Colab secrets")
            elif "429" in error_msg:
                print("  → Rate limited, wait a few seconds")
            elif "insufficient_quota" in error_msg.lower():
                print("  → Your OpenAI account has insufficient credits")

            return {
                "suggested_title": "Error",
                "confidence_score": 0.0,
                "model": self.model,
                "reasoning": f"API error: {str(e)}",
                "title_quality": "error"
            }

    def _validate_title_quality(self, title: str) -> str:
        """
        Validate if title meets quality standards.

        Returns:
            Quality flag: 'specific', 'generic_single_word', 'too_short', or 'ok'
        """
        if not title or title.lower() == 'error':
            return 'error'

        words = title.strip().lower().split()
        generic_words = {'analyst', 'specialist', 'coordinator', 'manager',
                        'director', 'teacher', 'technician', 'associate',
                        'assistant', 'supervisor', 'clerk'}

        # Single generic word = FAIL
        if len(words) == 1 and words[0] in generic_words:
            return 'generic_single_word'

        # Too short (less than 2 words)
        if len(words) < 2:
            return 'too_short'

        # Has generic word WITH qualifier = GOOD
        if any(w in generic_words for w in words) and len(words) >= 2:
            return 'specific'

        # No generic word but 2+ words = probably OK
        if len(words) >= 2:
            return 'specific'

        return 'ok'

# Initialize Auditor 1
print("\n" + "="*70)
print("INITIALIZING AUDITOR 1: GPT-4o")
print("="*70)

if not OPENAI_API_KEY:
    print("⚠ WARNING: OPENAI_API_KEY not set. Auditor 1 will not function.")
    print("  Add OPENAI_API_KEY to Colab secrets.")
    auditor1 = None
else:
    auditor1 = GPT4oAuditor(
        api_key=OPENAI_API_KEY,
        model=AUDITOR1_MODEL,
        temperature=AUDITOR1_TEMPERATURE
    )
    print("\n✓ Auditor 1 ready for processing")



INITIALIZING AUDITOR 1: GPT-4o
Initializing GPT-4o Title Auditor: gpt-4o...
✓ GPT-4o auditor initialized

✓ Auditor 1 ready for processing


In [ ]:
# Test Auditor 1 on first job
if auditor1:
    print("Testing Auditor 1 on sample job description...\n")
    test_result = auditor1.generate_title(jd_df['combined_jd_text'].iloc[0])

    print("="*70)
    print("AUDITOR 1 TEST RESULT")
    print("="*70)
    print(f"Original Title: {jd_df['original_title'].iloc[0]}")
    print(f"Suggested Title: {test_result['suggested_title']}")
    print(f"Confidence: {test_result['confidence_score']:.2%}")
    print(f"Title Quality: {test_result['title_quality']}")
    print(f"Reasoning: {test_result['reasoning']}")
    print(f"Model: {test_result['model']}")
else:
    print("⚠ Skipping test - Auditor 1 not initialized")


Testing Auditor 1 on sample job description...

AUDITOR 1 TEST RESULT
Original Title: Compensation Strategy Analyst I
Suggested Title: Compensation Analyst
Confidence: 95.00%
Title Quality: specific
Reasoning: The duties focus on developing and implementing compensation structures, conducting salary surveys, and evaluating job classifications, which are core responsibilities of a Compensation Analyst. The role involves analyzing competitive salary information and ensuring compensation programs are market-competitive, aligning with industry standards for this title.
Model: gpt-4o


## 4. Auditor 2: Claude Role Mapper (Enhanced)

**Enhancements implemented:**
- ✅ Multi-level classification (Major Role + Functional Subdomain)
- ✅ Integer confidence scores (1-100 instead of 0.0-1.0)
- ✅ Seniority consistency validation
- ✅ Temperature increased to 0.4 for better variance


In [ ]:
# =============================================================================
# AUDITOR 2: CLAUDE ROLE MAPPER (ENHANCED)
# With multi-level mapping, better confidence scoring, and consistency checks
# =============================================================================

class EnhancedClaudeRoleAuditor:
    """
    Auditor 2: Maps suggested titles to MNPS role groups with enhanced logic.

    Improvements:
    - Multi-level mapping (Major Role + Functional Subdomain)
    - Integer confidence scores (1-100 instead of 0.0-1.0)
    - Seniority consistency checks
    """

    def __init__(self, api_key: str, model: str, role_groups: Dict[str, str]):
        """
        Initialize Claude auditor.

        Args:
            api_key: Anthropic API key
            model: Claude model identifier
            role_groups: Dictionary of MNPS role groups and descriptions
        """
        self.client = anthropic.Anthropic(api_key=api_key)
        self.model = model
        self.role_groups = role_groups
        print(f"✓ Enhanced Claude Role Auditor initialized with {model}")

    def map_to_role_group(self, suggested_title: str, job_description: str) -> Dict:
        """
        Map a suggested title to MNPS role group with multi-level classification.

        Args:
            suggested_title: Title suggested by Auditor 1
            job_description: Original job description text

        Returns:
            Dict with major_role_group, functional_subdomain, confidence, reasoning, alignment
        """
        # Create role groups reference
        role_groups_text = "\n".join([
            f"- {role}: {desc}"
            for role, desc in self.role_groups.items()
        ])

        # Define common functional subdomains
        functional_domains = """
Common Functional Subdomains (examples):
- Financial/Accounting
- Human Resources
- Information Technology
- Facilities/Operations
- Student Services
- Instructional/Academic
- Compliance/Regulatory
- Communications/Marketing
- Data/Analytics
- Procurement/Purchasing
- Transportation
- Food Services
- Special Education
- Health/Medical
"""

        # Define seniority hierarchy
        seniority_hierarchy = """
Seniority Hierarchy (from highest to lowest):
1. Chief
2. Executive Director
3. Deputy Chief
4. Director
5. Manager
6. Supervisor
7. Coordinator
8. Specialist/Analyst (equal level)
9. Associate
10. Assistant
11. Aide/Paraprofessional
"""

        prompt = f"""You are an expert in job classification for Metro Nashville Public Schools (MNPS).

A job title has been suggested by an external auditor: "{suggested_title}"

Based on the job description below and the suggested title, perform a TWO-LEVEL classification:

LEVEL 1: MNPS Major Role Group (choose ONE from the list below)
LEVEL 2: Functional Subdomain (identify the primary functional area)

MNPS Major Role Groups:
{role_groups_text}

{functional_domains}

{seniority_hierarchy}

Job Description (first 3000 chars):
{job_description[:3000]}

CRITICAL INSTRUCTIONS:

1. MULTI-LEVEL MAPPING:
   - Choose the most appropriate Major Role Group
   - Identify the Functional Subdomain based on primary duties
   - Create a Full Classification combining both (e.g., "Financial Analyst")

2. CONFIDENCE SCORING:
   - Provide an INTEGER from 1-100 (NOT a decimal 0.0-1.0)
   - Be realistic - not all jobs should be 90-95
   - Consider ambiguity in the job description
   - Lower confidence (60-75) is acceptable for ambiguous cases

3. SENIORITY CONSISTENCY CHECK:
   - Compare the seniority level in "{suggested_title}" against your chosen Major Role Group
   - If they represent DIFFERENT hierarchical levels (e.g., "Coordinator" title mapped to "Director" group):
     * Alignment MUST be 'low'
     * You MUST explain this seniority mismatch in your reasoning
   - If they represent the SAME or SIMILAR levels:
     * Alignment can be 'high' or 'medium'

4. ALIGNMENT LEVELS:
   - high: Title and role group are well-aligned in both function AND seniority
   - medium: Aligned in function but different specificity level
   - low: Misaligned in function OR seniority mismatch

Respond in JSON format (no markdown):
{{
    "major_role_group": "[exact role name from MNPS list]",
    "functional_subdomain": "[primary functional area]",
    "full_classification": "[subdomain + major role, e.g., 'Financial Analyst']",
    "confidence_score": 85,
    "alignment_with_auditor1": "high/medium/low",
    "reasoning": "[brief explanation including seniority check if relevant]"
}}
"""

        try:
            message = self.client.messages.create(
                model=self.model,
                max_tokens=1000,
                temperature=AUDITOR2_TEMPERATURE,
                messages=[{"role": "user", "content": prompt}]
            )

            response_text = message.content[0].text

            # Extract JSON
            if '```json' in response_text:
                response_text = response_text.split('```json')[1].split('```')[0]
            elif '```' in response_text:
                response_text = response_text.split('```')[1].split('```')[0]

            result = json.loads(response_text.strip())

            # Validate confidence is integer
            if 'confidence_score' in result:
                if isinstance(result['confidence_score'], float):
                    # Convert to integer if it came as decimal
                    result['confidence_score'] = int(result['confidence_score'] * 100)

            result['model'] = self.model

            return result

        except Exception as e:
            print(f"Error in Claude role mapping: {e}")
            return {
                "major_role_group": "Error",
                "functional_subdomain": "Unknown",
                "full_classification": "Error",
                "confidence_score": 0,
                "alignment_with_auditor1": "unknown",
                "model": self.model,
                "reasoning": f"Error: {str(e)}"
            }

# Initialize Auditor 2
print("\n" + "="*70)
print("INITIALIZING AUDITOR 2: ENHANCED CLAUDE")
print("="*70)

if not ANTHROPIC_API_KEY:
    print("⚠ WARNING: ANTHROPIC_API_KEY not set. Auditor 2 will not function.")
    print("  Add ANTHROPIC_API_KEY to Colab secrets.")
    auditor2 = None
else:
    auditor2 = EnhancedClaudeRoleAuditor(
        api_key=ANTHROPIC_API_KEY,
        model=AUDITOR2_MODEL,
        role_groups=ROLE_GROUPS
    )
    print("\n✓ Auditor 2 ready for processing")



INITIALIZING AUDITOR 2: ENHANCED CLAUDE
✓ Enhanced Claude Role Auditor initialized with claude-sonnet-4-20250514

✓ Auditor 2 ready for processing


In [ ]:
# Test Auditor 2 on first job
if auditor2:
    print("Testing Auditor 2 on sample job...\n")
    test_result2 = auditor2.map_to_role_group(
        test_result['suggested_title'],
        jd_df['combined_jd_text'].iloc[0]
    )
    print(f"Suggested Title: {test_result['suggested_title']}")
    print(f"Suggested Role Group: {test_result2['major_role_group']}")
    print(f"Confidence: {test_result2['confidence_score']:.2f}")
    print(f"Alignment: {test_result2['alignment_with_auditor1']}")
    print(f"Reasoning: {test_result2['reasoning']}")
else:
    print("⚠ Skipping Auditor 2 test - API key not configured")

Testing Auditor 2 on sample job...

Suggested Title: Compensation Analyst
Suggested Role Group: Analyst
Confidence: 92.00
Alignment: high
Reasoning: The job description clearly aligns with the Analyst role group, focusing on data analysis, research, and strategic decision support for compensation matters. Core duties include analyzing salary data, conducting compensation surveys, evaluating job classifications, and making recommendations based on quantitative analysis. The functional area is definitively Human Resources, specifically compensation and classification. The suggested title 'Compensation Analyst' aligns perfectly with both the analytical nature of the work and the seniority level, as Analyst positions are at the appropriate level (8th in hierarchy) for this type of specialized analytical work requiring 1-3 years of experience.


## 5. Full Pipeline Processing

In [ ]:
def process_audit_pipeline(df: pd.DataFrame, auditor1, auditor2, batch_size: int = 10) -> List[Dict]:
    """
    Process all job descriptions through both auditors.

    Args:
        df: DataFrame with job descriptions
        auditor1: JobBERT auditor instance
        auditor2: LLM auditor instance
        batch_size: Size of batches for progress tracking

    Returns:
        List of audit results
    """
    results = []

    print(f"Processing {len(df)} job descriptions...\n")

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Auditing jobs"):
        job_desc = row['combined_jd_text']

        # Auditor 1: Generate title
        auditor1_result = auditor1.generate_title(job_desc)

        # Auditor 2: Map to role group
        if auditor2:
            auditor2_result = auditor2.map_to_role_group(
                auditor1_result['suggested_title'],
                job_desc
            )
        else:
            auditor2_result = {
                "suggested_role_group": "Not processed",
                "confidence_score": 0.0,
                "alignment_with_auditor1": "unknown",
                "model": "None",
                "reasoning": "Auditor 2 not configured"
            }

        # Combine results
        result = {
            "job_index": int(idx),
            "job_code": row.get('Job Code', ''),
            "original_title": row.get('original_title', ''),
            "auditor1": auditor1_result,
            "auditor2": auditor2_result,
            "timestamp": datetime.now().isoformat()
        }

        results.append(result)

        # Save intermediate results
        if SAVE_INTERMEDIATE and (idx + 1) % batch_size == 0:
            temp_file = os.path.join(OUTPUT_DIR, f"audit_progress_{idx+1}.json")
            with open(temp_file, 'w') as f:
                json.dump(results, f, indent=2)

    return results

In [ ]:
# Run full audit pipeline
audit_results = process_audit_pipeline(jd_df, auditor1, auditor2, BATCH_SIZE)

Processing 67 job descriptions...



Auditing jobs:   0%|          | 0/67 [00:00<?, ?it/s]

## 6. Output Generation

In [ ]:
# Save final results with enhanced multi-level classification
if 'audit_results' in locals():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = os.path.join(OUTPUT_DIR, f"auditor_results_enhanced_{timestamp}.json")

    with open(output_file, 'w') as f:
        json.dump(audit_results, f, indent=2)

    print(f"\n✓ Full results saved to: {output_file}")
    print(f"  Total jobs processed: {len(audit_results)}")

    # Also save as CSV for easier viewing
    csv_file = output_file.replace('.json', '.csv')
    results_df = pd.DataFrame([{
        'job_index': r['job_index'],
        'job_code': r['job_code'],
        'original_title': r['original_title'],
        'auditor1_suggested_title': r['auditor1'].get('suggested_title', 'N/A'),
        'auditor1_confidence': r['auditor1'].get('confidence_score', 0.0),
        'auditor1_title_quality': r['auditor1'].get('title_quality', 'unknown'),
        'auditor2_major_role': r['auditor2'].get('major_role_group', r['auditor2'].get('suggested_role_group', 'N/A')),
        'auditor2_subdomain': r['auditor2'].get('functional_subdomain', 'N/A'),
        'auditor2_full_classification': r['auditor2'].get('full_classification', 'N/A'),
        'auditor2_confidence': r['auditor2'].get('confidence_score', 0),
        'auditor2_alignment': r['auditor2'].get('alignment_with_auditor1', 'unknown'),
        'auditor1_model': r['auditor1'].get('model', 'unknown'),
        'auditor2_model': r['auditor2'].get('model', 'unknown'),
        'timestamp': r.get('timestamp', '')
    } for r in audit_results])

    results_df.to_csv(csv_file, index=False)
    print(f"✓ CSV summary saved to: {csv_file}")

    # Enhanced quality assessment - capture to string for summary file
    summary_lines = []
    summary_lines.append("="*70)
    summary_lines.append("ENHANCED RESULTS SUMMARY")
    summary_lines.append("="*70)

    summary_lines.append("\n1. AUDITOR 1 (GPT-4o) - Title Quality:")
    quality_counts = results_df['auditor1_title_quality'].value_counts()
    for quality, count in quality_counts.items():
        pct = count / len(results_df) * 100
        icon = "✓" if quality == 'specific' else "⚠" if quality == 'generic_single_word' else ""
        line = f"   {icon} {quality}: {count} ({pct:.1f}%)"
        summary_lines.append(line)
        print(line)

    summary_lines.append("\n2. AUDITOR 2 (Claude) - Multi-Level Classification:")
    line = f"   Unique Major Roles: {results_df['auditor2_major_role'].nunique()}"
    summary_lines.append(line)
    print(line)

    if 'auditor2_subdomain' in results_df.columns and results_df['auditor2_subdomain'].notna().any():
        line = f"   Unique Subdomains: {results_df['auditor2_subdomain'].nunique()}"
        summary_lines.append(line)
        print(line)
        line = f"   Unique Full Classifications: {results_df['auditor2_full_classification'].nunique()}"
        summary_lines.append(line)
        print(line)

    summary_lines.append("\n3. Confidence Score Distribution:")
    summary_lines.append("   Auditor 1 (GPT-4o):")
    line = f"     Mean: {results_df['auditor1_confidence'].mean():.1f}%"
    summary_lines.append(line)
    print(line)
    line = f"     Min: {results_df['auditor1_confidence'].min():.1f}%"
    summary_lines.append(line)
    print(line)
    line = f"     Max: {results_df['auditor1_confidence'].max():.1f}%"
    summary_lines.append(line)
    print(line)
    line = f"     Std Dev: {results_df['auditor1_confidence'].std():.1f}"
    summary_lines.append(line)
    print(line)

    summary_lines.append("\n   Auditor 2 (Claude):")
    line = f"     Mean: {results_df['auditor2_confidence'].mean():.1f}"
    summary_lines.append(line)
    print(line)
    line = f"     Min: {results_df['auditor2_confidence'].min():.0f}"
    summary_lines.append(line)
    print(line)
    line = f"     Max: {results_df['auditor2_confidence'].max():.0f}"
    summary_lines.append(line)
    print(line)
    line = f"     Std Dev: {results_df['auditor2_confidence'].std():.1f}"
    summary_lines.append(line)
    print(line)

    summary_lines.append("\n4. Alignment Distribution:")
    alignment_counts = results_df['auditor2_alignment'].value_counts()
    for alignment, count in alignment_counts.items():
        pct = count / len(results_df) * 100
        line = f"   {alignment}: {count} ({pct:.1f}%)"
        summary_lines.append(line)
        print(line)

    # Show top functional subdomains if available
    if 'auditor2_subdomain' in results_df.columns and results_df['auditor2_subdomain'].notna().any():
        summary_lines.append("\n5. Top 10 Functional Subdomains:")
        print("\n5. Top 10 Functional Subdomains:")
        for subdomain, count in results_df['auditor2_subdomain'].value_counts().head(10).items():
            line = f"   • {subdomain}: {count}"
            summary_lines.append(line)
            print(line)

    # Show sample with new fields
    print(f"\n" + "="*70)
    print("SAMPLE RESULTS (first 10)")
    print("="*70)
    if 'auditor2_subdomain' in results_df.columns:
        display_cols = ['original_title', 'auditor1_suggested_title',
                        'auditor2_major_role', 'auditor2_subdomain',
                        'auditor2_alignment', 'auditor1_confidence', 'auditor2_confidence']
    else:
        display_cols = ['original_title', 'auditor1_suggested_title',
                        'auditor2_major_role', 'auditor2_alignment',
                        'auditor1_confidence', 'auditor2_confidence']
    print(results_df[display_cols].head(10).to_string(index=False))

    # Flag low alignment jobs
    if 'auditor2_alignment' in results_df.columns:
        low_alignment = results_df[results_df['auditor2_alignment'] == 'low']
        if len(low_alignment) > 0:
            print(f"\n" + "="*70)
            print(f"⚠ FLAGGED: {len(low_alignment)} jobs with LOW alignment")
            print("="*70)
            summary_lines.append(f"\n⚠ FLAGGED: {len(low_alignment)} jobs with LOW alignment")
            summary_lines.append("="*70)
            print("These may have seniority mismatches or functional disagreements:")
            summary_lines.append("These may have seniority mismatches or functional disagreements:")
            for idx, row in low_alignment.head(5).iterrows():
                print(f"\n{idx+1}. {row['original_title']}")
                print(f"   Auditor 1: {row['auditor1_suggested_title']}")
                print(f"   Auditor 2: {row['auditor2_major_role']}")
                summary_lines.append(f"\n{idx+1}. {row['original_title']}")
                summary_lines.append(f"   Auditor 1: {row['auditor1_suggested_title']}")
                summary_lines.append(f"   Auditor 2: {row['auditor2_major_role']}")
                if 'auditor2_subdomain' in row and pd.notna(row['auditor2_subdomain']):
                    print(f"   Subdomain: {row['auditor2_subdomain']}")
                    print(f"   Full: {row['auditor2_full_classification']}")
                    summary_lines.append(f"   Subdomain: {row['auditor2_subdomain']}")
                    summary_lines.append(f"   Full: {row['auditor2_full_classification']}")

    # Show provenance
    print(f"\n" + "="*70)
    print("PROVENANCE")
    print("="*70)
    summary_lines.append("\n" + "="*70)
    summary_lines.append("PROVENANCE")
    summary_lines.append("="*70)
    line = f"Auditor 1 Model: {results_df['auditor1_model'].iloc[0]}"
    summary_lines.append(line)
    print(line)
    line = f"Auditor 2 Model: {results_df['auditor2_model'].iloc[0]}"
    summary_lines.append(line)
    print(line)
    line = f"First job timestamp: {results_df['timestamp'].iloc[0]}"
    summary_lines.append(line)
    print(line)
    line = f"Last job timestamp: {results_df['timestamp'].iloc[-1]}"
    summary_lines.append(line)
    print(line)
    line = f"Notebook version: {NOTEBOOK_VERSION}"
    summary_lines.append(line)
    print(line)

    # Save summary to text file
    summary_file = os.path.join(OUTPUT_DIR, f"audit_summary_{timestamp}.txt")
    with open(summary_file, 'w') as f:
        f.write('\n'.join(summary_lines))
    print(f"\n✓ Audit summary saved to: {summary_file}")

    # Create zip file with all outputs
    import zipfile
    zip_file = os.path.join(OUTPUT_DIR, f"audit_results_{timestamp}.zip")
    with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(output_file, os.path.basename(output_file))
        zipf.write(csv_file, os.path.basename(csv_file))
        zipf.write(summary_file, os.path.basename(summary_file))

    print(f"\n✓ All files packaged in: {zip_file}")

    # Download files in Colab
    print(f"\n" + "="*70)
    print("DOWNLOADS")
    print("="*70)

    try:
        from google.colab import files

        print("\n📦 Downloading ZIP file (contains all results)...")
        files.download(zip_file)
        print("   ✓ ZIP downloaded")

        print("\n📄 Downloading Audit Summary (text file)...")
        files.download(summary_file)
        print("   ✓ Summary downloaded")

        print("\n✅ All files ready!")
        print(f"\nZIP contains:")
        print(f"  • {os.path.basename(output_file)} (full JSON)")
        print(f"  • {os.path.basename(csv_file)} (CSV summary)")
        print(f"  • {os.path.basename(summary_file)} (audit summary)")

    except ImportError:
        print("\n⚠ Not running in Colab - files saved to outputs/ directory")
        print(f"\nFiles created:")
        print(f"  • {output_file}")
        print(f"  • {csv_file}")
        print(f"  • {summary_file}")
        print(f"  • {zip_file}")



✓ Full results saved to: outputs/auditor_results_enhanced_20260101_154757.json
  Total jobs processed: 67
✓ CSV summary saved to: outputs/auditor_results_enhanced_20260101_154757.csv
   ✓ specific: 67 (100.0%)
   Unique Major Roles: 21
   Unique Subdomains: 16
   Unique Full Classifications: 47
     Mean: 0.9%
     Min: 0.9%
     Max: 0.9%
     Std Dev: 0.0
     Mean: 89.9
     Min: 75
     Max: 95
     Std Dev: 5.2
   high: 64 (95.5%)
   low: 2 (3.0%)
   medium: 1 (1.5%)

5. Top 10 Functional Subdomains:
   • Financial/Accounting: 12
   • Instructional/Academic: 11
   • Facilities/Operations: 7
   • Student Services: 6
   • Compliance/Regulatory: 4
   • Human Resources: 4
   • Health/Medical: 3
   • Communications/Marketing: 3
   • Procurement/Purchasing: 3
   • Information Technology: 3

SAMPLE RESULTS (first 10)
                         original_title         auditor1_suggested_title auditor2_major_role        auditor2_subdomain auditor2_alignment  auditor1_confidence  auditor2_con

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✓ ZIP downloaded

📄 Downloading Audit Summary (text file)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✓ Summary downloaded

✅ All files ready!

ZIP contains:
  • auditor_results_enhanced_20260101_154757.json (full JSON)
  • auditor_results_enhanced_20260101_154757.csv (CSV summary)
  • audit_summary_20260101_154757.txt (audit summary)


In [ ]:
results_df['auditor2_alignment'] = [r['auditor2'].get('alignment_with_auditor1', 'unknown') for r in audit_results]
# Display summary statistics
print("\n" + "="*60)
print("AUDIT SUMMARY")
print("="*60)

print(f"\nAuditor 1 (JobBERT v2) Results:")
print(f"  - Unique titles suggested: {results_df['auditor1_suggested_title'].nunique()}")
print(f"  - Average confidence: {results_df['auditor1_confidence'].mean():.2%}")

if auditor2:
    print(f"\nAuditor 2 ({AUDITOR2_MODEL}) Results:")
    print(f"  - Role groups assigned:")
    for role, count in results_df['auditor2_major_role'].value_counts().items():
        print(f"    • {role}: {count}")
    print(f"  - Average confidence: {results_df['auditor2_confidence'].mean():.2%}")
    print(f"  - Alignment distribution:")
    for alignment, count in results_df['auditor2_alignment'].value_counts().items():
        print(f"    • {alignment}: {count}")

print(f"\n✓ External audit pipeline complete!")
print(f"  Results ready for Reporter analysis.")


AUDIT SUMMARY

Auditor 1 (JobBERT v2) Results:
  - Unique titles suggested: 57
  - Average confidence: 92.10%

Auditor 2 (claude-sonnet-4-20250514) Results:
  - Role groups assigned:
    • Specialist: 22
    • Coordinator: 6
    • Teacher: 6
    • Analyst: 5
    • Manager: 4
    • Director: 3
    • Technician: 3
    • Paraprofessional: 2
    • Engineer: 2
    • Auditor: 2
    • Chief: 2
    • Clerk: 1
    • Driver: 1
    • Architect (Technology-Focused): 1
    • Principal: 1
    • Therapist: 1
    • Skilled Laborer: 1
    • Secretary: 1
    • Counselor: 1
    • Assistant Principal: 1
    • Supervisor: 1
  - Average confidence: 8989.55%
  - Alignment distribution:
    • high: 64
    • low: 2
    • medium: 1

✓ External audit pipeline complete!
  Results ready for Reporter analysis.


## Summary of Enhancements

This notebook includes ALL colleague feedback and patches:

### ✅ Patch 1: Multi-Level Role Mapping
- **What:** Auditor 2 now returns `major_role_group`, `functional_subdomain`, and `full_classification`
- **Why:** Enables detection of functional domain drift
- **Impact:** Can distinguish "Compensation Analyst" from "IT Analyst"

### ✅ Patch 2: Confidence Variance
- **What:** Temperature increased to 0.4, integer scores (1-100)
- **Why:** Eliminates "confidence pegging" at 90/95
- **Impact:** More realistic risk assessment

### ✅ Patch 3: Seniority Consistency
- **What:** Automatic validation of seniority levels
- **Why:** Prevents illogical mappings (e.g., "Coordinator" → "Director")
- **Impact:** Flags low alignment when seniority mismatches

### ✅ Patch 4: Provenance Fix
- **What:** Corrected key names for model and timestamp
- **Why:** Model names were showing as "unknown"
- **Impact:** Full audit trail now working

---

**Expected Results:**
- Unique titles: 50-60 (vs 6 in v2)
- Title quality: 100% specific (vs 0% in v2)
- Confidence variance: Range 68-96 (vs 90/95 only)
- Functional subdomains: 15-20 unique
- Seniority mismatches: Auto-flagged

**Ready for production use!**